In [1]:
!export PYTORCH_CUDA_ALLOC_CONF=max_split_size_mb:128

In [7]:
# --- ОБЩИЕ ИМПОРТЫ ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import numpy as np
import re  # Импортируем библиотеку для регулярных выражений
import os
from pathlib import Path
from PIL import Image
import torchvision.transforms as T
import torchvision.models as models
try:
    import trimesh
    print("Библиотека trimesh успешно импортирована.")
except ImportError:
    print("!!! Trimesh не найден. Пожалуйста, установите: pip install trimesh[easy] !!!")


# --- 1. ЗАГРУЗЧИК STL И ДАТАСЕТ ---

def load_stl_xyz_only(stl_path, num_points=1024):
    """(Без изменений) Загружает, сэмплирует, центрирует и нормализует STL."""
    try:
        mesh = trimesh.load(stl_path, process=True, force='mesh')
        if isinstance(mesh, trimesh.Scene): mesh = mesh.dump(concatenate=True)
        if not hasattr(mesh, 'vertices') or mesh.vertices.shape[0] == 0: return None
        points, _ = trimesh.sample.sample_surface(mesh, num_points * 2)
        if len(points) < num_points:
            indices = np.random.choice(len(points), num_points, replace=True)
        else:
            indices = np.random.choice(len(points), num_points, replace=False)
        points = points[indices]
        centroid = np.mean(points, axis=0)
        points -= centroid
        max_dist = np.max(np.linalg.norm(points, axis=1))
        if max_dist < 1e-6: return None
        points /= max_dist
        return points.astype(np.float32)
    except Exception:
        return None

class PairedStlImageDataset(Dataset):
    """(ПЕРЕРАБОТАНО) Датасет для структуры 1 STL -> 25 изображений."""
    def __init__(self, stl_root, image_root, num_points=4096, image_size=224):
        self.num_points = num_points
        self.stl_root = Path(stl_root)
        self.image_root = Path(image_root)
        all_stl_files = sorted([f for f in self.stl_root.rglob('*.stl') if f.is_file()])
        
        self.paired_files = []
        # Паттерн для поиска 4 цифр в имени файла
        four_digit_pattern = re.compile(r'(\d{4})')

        for stl_path in tqdm(all_stl_files, desc="Сопоставление файлов"):
            match = four_digit_pattern.search(stl_path.stem)
            if not match:
                continue
            
            stl_number = match.group(1) # Извлекаем 4-значный номер
            
            # Ищем все изображения, начинающиеся с этого номера
            # (например, '0001_00.png', '0001_01.png', ...)
            image_paths = sorted(self.image_root.glob(f"{stl_number}_*.png"))
            
            for image_path in image_paths:
                self.paired_files.append((stl_path, image_path))
        
        print(f"\nНайдено {len(all_stl_files)} STL файлов.")
        print(f"Создано {len(self.paired_files)} пар (STL, Изображение) для обучения.")
        
        self.image_transform = T.Compose([
            T.Resize((image_size, image_size)), T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self): return len(self.paired_files)

    def __getitem__(self, idx):
        stl_path, image_path = self.paired_files[idx]
        points = load_stl_xyz_only(stl_path, self.num_points)
        if points is None: return None
        try:
            image = Image.open(image_path).convert("RGB")
            image_tensor = self.image_transform(image)
        except Exception: return None
        return torch.from_numpy(points), image_tensor

def paired_collate_fn(batch):
    """(Без изменений) Collate функция для нового датасета."""
    batch = list(filter(lambda x: x is not None, batch))
    if not batch: return None, None
    points, images = zip(*batch)
    return torch.stack(points), torch.stack(images)

# --- 2. АРХИТЕКТУРА МОДЕЛЕЙ ---

# 2.1 STL ЭНКОДЕР (PointNet++ из вашего кода)

# (ИСПРАВЛЕНО) Вспомогательные функции для PointNet++ в читаемом и рабочем виде
def farthest_point_sample(xyz, npoint):
    device = xyz.device
    B, N, C = xyz.shape
    centroids = torch.zeros(B, npoint, dtype=torch.long).to(device)
    distance = torch.ones(B, N).to(device) * 1e10
    farthest = torch.randint(0, N, (B,), dtype=torch.long).to(device)
    batch_indices = torch.arange(B, dtype=torch.long).to(device)
    for i in range(npoint):
        centroids[:, i] = farthest
        centroid = xyz[batch_indices, farthest, :].view(B, 1, 3)
        dist = torch.sum((xyz - centroid) ** 2, -1)
        mask = dist < distance
        distance[mask] = dist[mask]
        farthest = torch.max(distance, -1)[1]
    return centroids

def query_ball_point(radius, nsample, xyz, new_xyz):
    device = xyz.device
    B, N, C = xyz.shape
    _, S, _ = new_xyz.shape
    group_idx = torch.arange(N, dtype=torch.long, device=device).view(1, 1, N).repeat(B, S, 1)
    sqrdists = torch.sum((xyz.unsqueeze(1) - new_xyz.unsqueeze(2)) ** 2, -1)
    group_idx[sqrdists > radius ** 2] = N
    group_idx = group_idx.sort(dim=-1)[0][:, :, :nsample]
    group_first = group_idx[:, :, 0].view(B, S, 1).repeat(1, 1, nsample)
    mask = group_idx == N
    group_idx[mask] = group_first[mask]
    return group_idx

def index_points(points, idx):
    device = points.device
    B = points.shape[0]
    view_shape = list(idx.shape)
    view_shape[1:] = [1] * (len(view_shape) - 1)
    repeat_shape = list(idx.shape)
    repeat_shape[0] = 1
    batch_indices = torch.arange(B, dtype=torch.long).to(device).view(view_shape).repeat(repeat_shape)
    new_points = points[batch_indices, idx, :]
    return new_points

class PointNetSetAbstraction(nn.Module):
    def __init__(self, npoint, radius, nsample, in_channel, mlp, group_all):
        super(PointNetSetAbstraction, self).__init__()
        self.npoint, self.radius, self.nsample, self.group_all = npoint, radius, nsample, group_all
        self.mlp_convs, self.mlp_bns = nn.ModuleList(), nn.ModuleList()
        last_channel = in_channel + 3
        for out_channel in mlp:
            self.mlp_convs.append(nn.Conv2d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm2d(out_channel))
            last_channel = out_channel

    def forward(self, xyz, points):
        if not self.group_all:
            new_xyz_idx = farthest_point_sample(xyz, self.npoint)
            new_xyz = index_points(xyz, new_xyz_idx)
            group_idx = query_ball_point(self.radius, self.nsample, xyz, new_xyz)
            grouped_xyz = index_points(xyz, group_idx)
            grouped_xyz -= new_xyz.unsqueeze(2)
            if points is not None:
                grouped_points = index_points(points, group_idx)
                features = torch.cat([grouped_xyz, grouped_points], dim=-1)
            else:
                features = grouped_xyz
        else:
            new_xyz = torch.zeros(xyz.shape[0], 1, 3, device=xyz.device)
            grouped_xyz = xyz.view(xyz.shape[0], 1, -1, 3)
            if points is not None:
                features = torch.cat([grouped_xyz, points.view(points.shape[0], 1, -1, points.shape[2])], dim=-1)
            else:
                features = grouped_xyz
        
        features = features.permute(0, 3, 2, 1)
        for conv, bn in zip(self.mlp_convs, self.mlp_bns):
            features = F.relu(bn(conv(features)))
        
        new_points = torch.max(features, 2)[0].permute(0, 2, 1)
        return new_xyz, new_points

class StlEncoder(nn.Module):
    """(ИСПРАВЛЕНО) Ваш класс Encoder, переименован для ясности."""
    def __init__(self, in_features=3, embedding_dim=256):
        super().__init__()
        # in_channel теперь правильно 0, так как у нас нет доп. фичей кроме xyz
        self.sa1 = PointNetSetAbstraction(npoint=512, radius=0.2, nsample=32, in_channel=in_features-3, mlp=[64, 64, 128], group_all=False)
        self.sa2 = PointNetSetAbstraction(npoint=128, radius=0.4, nsample=64, in_channel=128, mlp=[128, 128, 256], group_all=False)
        self.sa3 = PointNetSetAbstraction(npoint=None, radius=None, nsample=None, in_channel=256, mlp=[256, 512, 1024], group_all=True)
        self.fc1 = nn.Linear(1024, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.drop1 = nn.Dropout(0.4)
        self.fc_embedding = nn.Linear(512, embedding_dim)
    
    def forward(self, xyz):
        l1_xyz, l1_points = self.sa1(xyz, points=None)
        l2_xyz, l2_points = self.sa2(l1_xyz, l1_points)
        _, l3_points = self.sa3(l2_xyz, l2_points)
        x = l3_points.view(xyz.shape[0], -1)
        x = self.drop1(F.relu(self.bn1(self.fc1(x))))
        embedding = self.fc_embedding(x)
        return F.normalize(embedding, dim=1)

# 2.2. НОВЫЙ ЭНКОДЕР ИЗОБРАЖЕНИЙ (без изменений)
class ImageEncoder(nn.Module):
    def __init__(self, embedding_dim=256):
        super().__init__()
        base_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(base_model.children())[:-1])
        self.projection = nn.Sequential(
            nn.Linear(base_model.fc.in_features, 512),
            nn.ReLU(),
            nn.Linear(512, embedding_dim)
        )
    def forward(self, x):
        h = self.features(x).view(x.shape[0], -1)
        return F.normalize(self.projection(h), dim=1)
    
# --- 3. ФУНКЦИЯ ПОТЕРЬ (без изменений) ---
class InfoNCELoss(nn.Module):
    def __init__(self,temperature=0.07): super().__init__();self.temperature=temperature;self.criterion=nn.CrossEntropyLoss()
    def forward(self,image_features,stl_features): logits=(image_features@stl_features.T)/self.temperature;labels=torch.arange(image_features.shape[0],device=image_features.device);return(self.criterion(logits,labels)+self.criterion(logits.T,labels))/2

# --- 4. ОБУЧАЮЩИЙ ЦИКЛ (без изменений) ---
def train_cross_modal(stl_encoder,image_encoder,dataloader,optimizer,loss_fn,device,epochs,path):
    stl_encoder.train();image_encoder.train();print("--- ЗАПУСК МЕЖМОДАЛЬНОГО КОНТРАСТИВНОГО ОБУЧЕНИЯ ---")
    for epoch in range(epochs):
        total_loss=0;num_batches=0;p_bar=tqdm(dataloader,desc=f"Epoch {epoch+1}/{epochs}",leave=True)
        for stl_batch,image_batch in p_bar:
            if stl_batch is None:continue
            stl_batch,image_batch=stl_batch.to(device),image_batch.to(device);optimizer.zero_grad()
            loss=loss_fn(image_encoder(image_batch),stl_encoder(stl_batch));loss.backward();optimizer.step()
            total_loss+=loss.item();num_batches+=1;p_bar.set_postfix(loss=f"{loss.item():.4f}")
        avg_loss=total_loss/num_batches if num_batches>0 else 0;print(f"Epoch {epoch+1}/{epochs} | Average Loss: {avg_loss:.4f}")
        torch.save({'stl_encoder_state_dict':stl_encoder.state_dict(),'image_encoder_state_dict':image_encoder.state_dict()},path)
    print(f"Обучение завершено. Модели сохранены в {path}")



Библиотека trimesh успешно импортирована.


In [4]:
# --- 5. ЗАПУСК ОБУЧЕНИЯ ---
if __name__ == '__main__':
    # --- ОСНОВНЫЕ ПАРАМЕТРЫ ---
    STL_DIR = "train/train_data/models"
    IMAGE_DIR = "train/train_data/images"
    MODEL_SAVE_PATH = "cross_modal_checkpoint.pth"
    BATCH_SIZE = 32 # Можно уменьшить, если не хватает видеопамяти
    LEARNING_RATE = 1e-4
    NUM_EPOCHS = 100
    NUM_POINTS = 4096
    EMBEDDING_DIM = 256
    IMAGE_SIZE = 224

    # --- НАСТРОЙКА ---
    Path(STL_DIR).mkdir(parents=True, exist_ok=True)
    Path(IMAGE_DIR).mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Используемое устройство: {device}")

    # --- ДАННЫЕ ---
    dataset = PairedStlImageDataset(STL_DIR, IMAGE_DIR, num_points=NUM_POINTS, image_size=IMAGE_SIZE)
    if len(dataset) == 0:
        print("\n!!! ВНИМАНИЕ: Датасет пуст. Проверьте пути и формат имен файлов.")
    else:
        # num_workers=0 для Windows, чтобы избежать проблем с мультипроцессингом
        dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, 
                                collate_fn=paired_collate_fn, num_workers=0, pin_memory=True)
        
        # --- МОДЕЛИ, ЛОСС, ОПТИМИЗАТОР ---
        stl_encoder = StlEncoder(embedding_dim=EMBEDDING_DIM).to(device)
        image_encoder = ImageEncoder(embedding_dim=EMBEDDING_DIM).to(device)
        loss_fn = InfoNCELoss(temperature=0.07)
        optimizer = torch.optim.AdamW(
            list(stl_encoder.parameters()) + list(image_encoder.parameters()),
            lr=LEARNING_RATE, weight_decay=1e-5
        )

        # --- ОБУЧЕНИЕ ---
        train_cross_modal(stl_encoder, image_encoder, dataloader, optimizer, 
                          loss_fn, device, NUM_EPOCHS, MODEL_SAVE_PATH)

Используемое устройство: cuda


Сопоставление файлов:  71%|███████   | 374/525 [00:01<00:00, 188.48it/s]


KeyboardInterrupt: 

In [5]:
# --- ПАРАМЕТРЫ ИЗ ПРЕДЫДУЩЕЙ ЯЧЕЙКИ ---
# Убедитесь, что они совпадают с параметрами обучения
# STL_DIR, IMAGE_DIR, MODEL_SAVE_PATH, NUM_POINTS, EMBEDDING_DIM, IMAGE_SIZE, device

print("--- ЗАГРУЗКА ОБУЧЕННЫХ МОДЕЛЕЙ ---")
# Инициализируем модели с той же архитектурой
stl_encoder = StlEncoder(embedding_dim=EMBEDDING_DIM).to(device)
image_encoder = ImageEncoder(embedding_dim=EMBEDDING_DIM).to(device)

# Загружаем сохраненные веса
checkpoint = torch.load(MODEL_SAVE_PATH, map_location=device)
stl_encoder.load_state_dict(checkpoint['stl_encoder_state_dict'])
image_encoder.load_state_dict(checkpoint['image_encoder_state_dict'])

stl_encoder.eval()
image_encoder.eval()
print("Модели успешно загружены и переведены в режим оценки (eval).")


def generate_stl_embeddings(encoder, stl_dir, num_points, device):
    stl_files = sorted([f for f in Path(stl_dir).rglob('*.stl') if f.is_file()])
    embeddings = {}
    with torch.no_grad():
        for path in tqdm(stl_files, desc="Генерация STL эмбеддингов"):
            points = load_stl_xyz_only(path, num_points)
            if points is not None:
                points_tensor = torch.from_numpy(points).unsqueeze(0).to(device)
                embedding = encoder(points_tensor).squeeze(0).cpu().numpy()
                embeddings[str(path)] = embedding
    return embeddings

def generate_image_embeddings(encoder, image_dir, image_size, device):
    img_files = sorted([f for f in Path(image_dir).rglob('*') if f.suffix.lower() in ['.png', '.jpg', '.jpeg']])
    transform = T.Compose([
        T.Resize((image_size, image_size)), T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    embeddings = {}
    with torch.no_grad():
        for path in tqdm(img_files, desc="Генерация Image эмбеддингов"):
            try:
                img = Image.open(path).convert("RGB")
                img_tensor = transform(img).unsqueeze(0).to(device)
                embedding = encoder(img_tensor).squeeze(0).cpu().numpy()
                embeddings[str(path)] = embedding
            except Exception: continue
    return embeddings

# --- ЗАПУСК ГЕНЕРАЦИИ ---
stl_embeddings_dict = generate_stl_embeddings(stl_encoder, STL_DIR, NUM_POINTS, device)
image_embeddings_dict = generate_image_embeddings(image_encoder, IMAGE_DIR, IMAGE_SIZE, device)

print(f"\nСгенерировано {len(stl_embeddings_dict)} STL эмбеддингов.")
print(f"Сгенерировано {len(image_embeddings_dict)} Image эмбеддингов.")

--- ЗАГРУЗКА ОБУЧЕННЫХ МОДЕЛЕЙ ---
Модели успешно загружены и переведены в режим оценки (eval).


Генерация Image эмбеддингов: 100%|██████████| 13650/13650 [03:10<00:00, 71.74it/s]


Сгенерировано 525 STL эмбеддингов.
Сгенерировано 13650 Image эмбеддингов.


In [27]:
# ===================== TEST MATCHING: STL ↔ группы рендеров gNNNN_* (по 26 кадров) =====================
# Результат: DataFrame assign_grouped (никаких CSV на диск не сохраняется)

from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

# --- Конфиг ---
TEST_STL_DIR      = "test/test_data/gallery_mesh_for_image"
TEST_IMG_DIR      = "test/test_data/gallery_image_for_mesh"
RENDER_PER_GROUP  = 26
SIM_THRESHOLD     = 0.25

# ---------- Вспомогательные ----------
def _to_numpy(x):
    try:
        import torch
        if isinstance(x, torch.Tensor):
            return x.detach().cpu().numpy()
    except Exception:
        pass
    return np.asarray(x)

def dict_to_matrix(d):
    if not isinstance(d, dict) or len(d) == 0:
        return np.empty((0,0), dtype=np.float32), []
    keys = list(d.keys())
    mats = []
    for k in keys:
        v = _to_numpy(d[k])
        if v.ndim == 1:
            v = v[None, :]
        mats.append(v.astype(np.float32, copy=False))
    return np.vstack(mats), keys

# --- 1) Генерация эмбеддингов для test ---
emb_stl_test = generate_stl_embeddings(stl_encoder, TEST_STL_DIR, NUM_POINTS, device)
emb_img_test = generate_image_embeddings(image_encoder, TEST_IMG_DIR, IMAGE_SIZE, device)

E_stl, stl_keys = dict_to_matrix(emb_stl_test)
E_img, img_keys = dict_to_matrix(emb_img_test)
print(f"[ok] STL={E_stl.shape}, IMG={E_img.shape}")

# --- 2) Группировка изображений по префиксу gNNNN_* ---
grp_regex = re.compile(r"^(g\d{4,6})[_-]")
group_to_indices = {}
for j, p in enumerate(img_keys):
    stem = Path(p).name
    m = grp_regex.match(stem)
    if not m:
        prefix = re.split(r"[_-]", stem, maxsplit=1)[0]
        if prefix.startswith('g') and prefix[1:].isdigit():
            gid = prefix
        else:
            continue
    else:
        gid = m.group(1)
    group_to_indices.setdefault(gid, []).append(j)

group_ids = sorted(group_to_indices.keys())
print(f"[ok] Найдено групп: {len(group_ids)}")

# --- 3) Косинусное сходство и Hungarian STL↔GROUP ---
cos_dist = cdist(E_stl, E_img, metric="cosine")
S = 1.0 - cos_dist

S_group = np.zeros((E_stl.shape[0], len(group_ids)), dtype=np.float32)
for gi, gid in enumerate(group_ids):
    idxs = group_to_indices[gid]
    S_group[:, gi] = S[:, idxs].mean(axis=1)

cost = 1.0 - S_group
row_ind, col_ind = linear_sum_assignment(cost)
group_to_stl_idx = { group_ids[j_grp]: int(i_stl) for i_stl, j_grp in zip(row_ind, col_ind) }

# --- 4) Формируем DataFrame назначений (только в памяти) ---
rows = []
for gid, idxs in group_to_indices.items():
    i_stl = group_to_stl_idx.get(gid, None)
    for j in idxs:
        rows.append({
            "image_path": img_keys[j],
            "stl_path":   stl_keys[i_stl] if i_stl is not None else None,
            "similarity": float(S[i_stl, j]) if i_stl is not None else np.nan,
            "group_id":   gid
        })

assign_grouped = pd.DataFrame(rows).sort_values(
    ["stl_path", "group_id", "similarity"], ascending=[True, True, False]
).reset_index(drop=True)

print("Пример результата:")
display(assign_grouped.head(10))


Генерация Image эмбеддингов: 100%|██████████| 2600/2600 [00:36<00:00, 71.49it/s]


[ok] STL=(100, 256), IMG=(2600, 256)
[ok] Найдено групп: 100
Пример результата:


,image_path,stl_path,similarity,group_id
0,test/test_data/gallery_image_for_mesh/g0069_16...,test/test_data/gallery_mesh_for_image/g0000.stl,0.919304,g0069
1,test/test_data/gallery_image_for_mesh/g0069_9.png,test/test_data/gallery_mesh_for_image/g0000.stl,0.917139,g0069
2,test/test_data/gallery_image_for_mesh/g0069_18...,test/test_data/gallery_mesh_for_image/g0000.stl,0.910190,g0069
3,test/test_data/gallery_image_for_mesh/g0069_24...,test/test_data/gallery_mesh_for_image/g0000.stl,0.906059,g0069
4,test/test_data/gallery_image_for_mesh/g0069_23...,test/test_data/gallery_mesh_for_image/g0000.stl,0.905682,g0069
5,test/test_data/gallery_image_for_mesh/g0069_21...,test/test_data/gallery_mesh_for_image/g0000.stl,0.904389,g0069
6,test/test_data/gallery_image_for_mesh/g0069_0.png,test/test_data/gallery_mesh_for_image/g0000.stl,0.903440,g0069
7,test/test_data/gallery_image_for_mesh/g0069_17...,test/test_data/gallery_mesh_for_image/g0000.stl,0.902866,g0069
8,test/test_data/gallery_image_for_mesh/g0069_25...,test/test_data/gallery_mesh_for_image/g0000.stl,0.902714,g0069
9,test/test_data/gallery_image_for_mesh/g0069_7.png,test/test_data/gallery_mesh_for_image/g0000.stl,0.902519,g0069


In [33]:
# ===================== Retrieval API (image->mesh и mesh->image) =====================
from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

# используем вспомогательные из твоего блока
# _to_numpy, dict_to_matrix уже определены выше

def _ensure_gallery_stl_embeddings(stl_dir: str | None, stl_embeddings: dict | None,
                                   num_points: int, device):
    """
    Возвращает (E_stl, stl_keys, stl_embeddings_dict), где:
      E_stl: np.ndarray [N_mesh, D]
      stl_keys: list[str] пути мешей в том же порядке
      stl_embeddings_dict: исходный словарь {path->vec}
    Если stl_embeddings не передан, считает по stl_dir с твоей generate_stl_embeddings.
    """
    if stl_embeddings is None:
        assert stl_dir is not None, "Нужно указать stl_dir или stl_embeddings."
        stl_embeddings = generate_stl_embeddings(stl_encoder, stl_dir, num_points, device)
    E_stl, stl_keys = dict_to_matrix(stl_embeddings)
    if E_stl.size == 0:
        raise RuntimeError("Галерея STL пуста — нет эмбеддингов.")
    return E_stl, stl_keys, stl_embeddings

def _ensure_gallery_img_embeddings(img_dir: str | None, img_embeddings: dict | None,
                                   image_size: int, device):
    """
    Возвращает (E_img, img_keys, img_embeddings_dict).
    Если img_embeddings не передан, считает по img_dir с твоей generate_image_embeddings.
    """
    if img_embeddings is None:
        assert img_dir is not None, "Нужно указать img_dir или img_embeddings."
        img_embeddings = generate_image_embeddings(image_encoder, img_dir, image_size, device)
    E_img, img_keys = dict_to_matrix(img_embeddings)
    if E_img.size == 0:
        raise RuntimeError("Галерея изображений пуста — нет эмбеддингов.")
    return E_img, img_keys, img_embeddings

def _group_indices_by_prefix(img_keys: list[str], render_per_group: int = 26):
    """
    Группируем по префиксу gNNNN (поддержка '_' и '-') как в основном блоке.
    Возвращает:
      group_ids: list[str]
      group_to_indices: dict[group_id -> list[int]]
    """
    grp_regex = re.compile(r"^(g\d{4,6})[_-]")
    group_to_indices = {}
    for j, p in enumerate(img_keys):
        stem = Path(p).name
        m = grp_regex.match(stem)
        if not m:
            prefix = re.split(r"[_-]", stem, maxsplit=1)[0]
            if prefix.startswith('g') and prefix[1:].isdigit():
                gid = prefix
            else:
                # пропустим файлы, не соответствующие паттерну
                continue
        else:
            gid = m.group(1)
        group_to_indices.setdefault(gid, []).append(j)
    group_ids = sorted(group_to_indices.keys())
    # не навязываем строго 26, но это инфо может пригодиться
    return group_ids, group_to_indices

def search_images_to_meshes(
    query_images_dir: str,
    *,
    gallery_stl_embeddings: dict | None = None,
    gallery_stl_dir: str | None = None,
    topk: int = 1,
    image_size: int | None = None,
    num_points: int | None = None,
    device=None,
    use_existing_gallery_from_globals: bool = True
) -> pd.DataFrame:
    """
    Поиск image -> mesh: для КАЖДОГО изображения из query_images_dir находим top-K мешей из галереи.
    Галерея по умолчанию = emb_stl_test из твоего блока. Иначе можно передать gallery_stl_dir / gallery_stl_embeddings.

    Возвращает DataFrame с колонками: image_path, stl_path, similarity, rank (1..K).
    """
    # дефолты из твоих глобалок
    img_sz = IMAGE_SIZE if image_size is None else image_size
    pts    = NUM_POINTS if num_points is None else num_points
    dev    = device if device is not None else globals().get("device", None)
    assert dev is not None, "device не найден."

    # ГАЛЕРЕЯ МЕШЕЙ: берем существующие emb_stl_test, если не передано иначе
    if use_existing_gallery_from_globals and gallery_stl_embeddings is None and gallery_stl_dir is None:
        gallery_stl_embeddings = globals().get("emb_stl_test", None)
    E_stl, stl_keys, gallery_stl_embeddings = _ensure_gallery_stl_embeddings(
        gallery_stl_dir, gallery_stl_embeddings, pts, dev
    )

    # ЗАПРОСНЫЕ ИЗОБРАЖЕНИЯ
    q_img_emb = generate_image_embeddings(image_encoder, query_images_dir, img_sz, dev)
    E_qimg, qimg_keys = dict_to_matrix(q_img_emb)
    if E_qimg.size == 0:
        raise RuntimeError("В папке запросов не удалось получить эмбеддинги изображений.")

    # СХОДСТВО: [N_mesh, N_qimg]
    if E_stl.shape[1] != E_qimg.shape[1]:
        raise ValueError(f"Разная размерность: mesh D={E_stl.shape[1]} vs image D={E_qimg.shape[1]}")
    S = 1.0 - cdist(E_stl, E_qimg, metric="cosine")

    # Top-K по столбцам
    topk = max(1, int(topk))
    idx_sorted = np.argsort(-S, axis=0)[:topk, :]
    rows = []
    for j in range(S.shape[1]):
        for r, i_stl in enumerate(idx_sorted[:, j], start=1):
            rows.append({
                "image_path": qimg_keys[j],
                "stl_path":   stl_keys[int(i_stl)],
                "similarity": float(S[int(i_stl), j]),
                "rank":       r
            })
    df = pd.DataFrame(rows).sort_values(["image_path", "rank"]).reset_index(drop=True)
    return df

def search_meshes_to_images(
    query_meshes_dir: str,
    *,
    gallery_image_embeddings: dict | None = None,
    gallery_image_dir: str | None = None,
    mode: str = "group",     # "group" → ищем по группам gNNNN (усредняя), "image" → по отдельным изображениям
    topk: int = 1,
    image_size: int | None = None,
    num_points: int | None = None,
    device=None,
    use_existing_gallery_from_globals: bool = True,
    render_per_group: int = 26
) -> pd.DataFrame:
    """
    Поиск mesh -> image/группа:
      - mode="group": возвращает топ-K групп gNNNN (усредняет сходство по всем кадрам группы).
      - mode="image": возвращает топ-K отдельных изображений.

    Галерея по умолчанию = emb_img_test из твоего блока. Иначе можно передать gallery_image_dir / gallery_image_embeddings.

    Возвращает DataFrame:
      * mode="group": columns = [stl_path, group_id, score, rank]
      * mode="image": columns = [stl_path, image_path, similarity, rank]
    """
    img_sz = IMAGE_SIZE if image_size is None else image_size
    pts    = NUM_POINTS if num_points is None else num_points
    dev    = device if device is not None else globals().get("device", None)
    assert dev is not None, "device не найден."

    # ГАЛЕРЕЯ ИЗОБРАЖЕНИЙ
    if use_existing_gallery_from_globals and gallery_image_embeddings is None and gallery_image_dir is None:
        gallery_image_embeddings = globals().get("emb_img_test", None)
    E_img, img_keys, gallery_image_embeddings = _ensure_gallery_img_embeddings(
        gallery_image_dir, gallery_image_embeddings, img_sz, dev
    )

    # ЗАПРОСНЫЕ МЕШИ
    q_stl_emb = generate_stl_embeddings(stl_encoder, query_meshes_dir, pts, dev)
    E_qstl, qstl_keys = dict_to_matrix(q_stl_emb)
    if E_qstl.size == 0:
        raise RuntimeError("В папке запросов не удалось получить эмбеддинги STL.")

    if E_img.shape[1] != E_qstl.shape[1]:
        raise ValueError(f"Разная размерность: image D={E_img.shape[1]} vs mesh D={E_qstl.shape[1]}")

    # СХОДСТВО: [N_qstl, N_img]  (удобно — строки = запросы)
    S = 1.0 - cdist(E_qstl, E_img, metric="cosine")

    topk = max(1, int(topk))
    rows = []

    if mode == "image":
        # по отдельным изображениям
        idx_sorted = np.argsort(-S, axis=1)[:, :topk]  # [N_qstl, topk]
        for i in range(S.shape[0]):  # по каждому запросному мешу
            for r, j_img in enumerate(idx_sorted[i], start=1):
                rows.append({
                    "stl_path":   qstl_keys[i],
                    "image_path": img_keys[int(j_img)],
                    "similarity": float(S[i, int(j_img)]),
                    "rank":       r
                })
        df = pd.DataFrame(rows).sort_values(["stl_path", "rank"]).reset_index(drop=True)
        return df

    elif mode == "group":
        # агрегируем изображения галереи по группам gNNNN
        group_ids, group_to_indices = _group_indices_by_prefix(img_keys, render_per_group)
        if not group_ids:
            raise RuntimeError("Не удалось сгруппировать галерею изображений по префиксу gNNNN_*.")

        # среднее сходство по кадрам группы → матрица [N_qstl, N_groups]
        S_group = np.zeros((S.shape[0], len(group_ids)), dtype=np.float32)
        for gi, gid in enumerate(group_ids):
            idxs = group_to_indices[gid]
            S_group[:, gi] = S[:, idxs].mean(axis=1)

        idx_sorted = np.argsort(-S_group, axis=1)[:, :topk]  # [N_qstl, topk]
        for i in range(S_group.shape[0]):
            for r, gi in enumerate(idx_sorted[i], start=1):
                rows.append({
                    "stl_path": qstl_keys[i],
                    "group_id": group_ids[int(gi)],
                    "score":    float(S_group[i, int(gi)]),
                    "rank":     r
                })
        df = pd.DataFrame(rows).sort_values(["stl_path", "rank"]).reset_index(drop=True)
        return df

    else:
        raise ValueError("mode должен быть 'image' или 'group'.")


In [ ]:
# 0) У тебя уже выполнен блок с emb_stl_test / emb_img_test (галерея на test/*)

# 1) Поиск image -> mesh (для папки с 100 произвольными запросными картинками)
df_i2m = search_images_to_meshes(
    "test/test_data/queries_image_to_mesh",
    topk=5  # вернём Top-3 меша на каждую картинку
)
display(df_i2m.head(10))

# 2) Поиск mesh -> image (топ-кадры из галереи test/images)
# 2a) по отдельным изображениям:
df_m2i = search_meshes_to_images(
    "test/test_data/queries_mesh_to_image",
    mode="image",
    topk=5
)
display(df_m2i.head(10))



Генерация Image эмбеддингов: 100%|██████████| 100/100 [00:01<00:00, 66.90it/s]


,image_path,stl_path,similarity,rank
0,test/test_data/queries_image_to_mesh/q0000.png,test/test_data/gallery_mesh_for_image/g0031.stl,0.617783,1
1,test/test_data/queries_image_to_mesh/q0000.png,test/test_data/gallery_mesh_for_image/g0072.stl,0.613555,2
2,test/test_data/queries_image_to_mesh/q0000.png,test/test_data/gallery_mesh_for_image/g0033.stl,0.501693,3
3,test/test_data/queries_image_to_mesh/q0001.png,test/test_data/gallery_mesh_for_image/g0039.stl,0.819642,1
4,test/test_data/queries_image_to_mesh/q0001.png,test/test_data/gallery_mesh_for_image/g0047.stl,0.511766,2
5,test/test_data/queries_image_to_mesh/q0001.png,test/test_data/gallery_mesh_for_image/g0014.stl,0.493433,3
6,test/test_data/queries_image_to_mesh/q0002.png,test/test_data/gallery_mesh_for_image/g0080.stl,0.688380,1
7,test/test_data/queries_image_to_mesh/q0002.png,test/test_data/gallery_mesh_for_image/g0090.stl,0.676430,2
8,test/test_data/queries_image_to_mesh/q0002.png,test/test_data/gallery_mesh_for_image/g0067.stl,0.658878,3
9,test/test_data/queries_image_to_mesh/q0003.png,test/test_data/gallery_mesh_for_image/g0018.stl,0.935866,1


Генерация STL эмбеддингов: 100%|██████████| 100/100 [00:15<00:00,  6.61it/s]


,stl_path,image_path,similarity,rank
0,test/test_data/queries_mesh_to_image/q0000.stl,test/test_data/gallery_image_for_mesh/g0058_6.png,0.884574,1
1,test/test_data/queries_mesh_to_image/q0000.stl,test/test_data/gallery_image_for_mesh/g0027_19...,0.817363,2
2,test/test_data/queries_mesh_to_image/q0000.stl,test/test_data/gallery_image_for_mesh/g0027_21...,0.815852,3
3,test/test_data/queries_mesh_to_image/q0000.stl,test/test_data/gallery_image_for_mesh/g0027_9.png,0.813387,4
4,test/test_data/queries_mesh_to_image/q0000.stl,test/test_data/gallery_image_for_mesh/g0027_0.png,0.809903,5
5,test/test_data/queries_mesh_to_image/q0001.stl,test/test_data/gallery_image_for_mesh/g0045_9.png,0.822673,1
6,test/test_data/queries_mesh_to_image/q0001.stl,test/test_data/gallery_image_for_mesh/g0045_0.png,0.771804,2
7,test/test_data/queries_mesh_to_image/q0001.stl,test/test_data/gallery_image_for_mesh/g0045_1.png,0.751595,3
8,test/test_data/queries_mesh_to_image/q0001.stl,test/test_data/gallery_image_for_mesh/g0045_19...,0.748267,4
9,test/test_data/queries_mesh_to_image/q0001.stl,test/test_data/gallery_image_for_mesh/g0045_6.png,0.747816,5


In [44]:
# ===================== Fill submission table using retrieval results =====================
from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
import json

# --- укажи директории с запросами ---
QUERY_IMAGES_DIR = "test/test_data/queries_image_to_mesh"   # 100 картинок
QUERY_MESHES_DIR = "test/test_data/queries_mesh_to_image"   # 100 STL

SUBMISSION_IN = "submission copy.csv"
SUBMISSION_OUT = "submission.csv"
TOPK_I2M = 5
TOPK_M2I = 5

def _to_numpy(x):
    try:
        import torch
        if isinstance(x, torch.Tensor):
            return x.detach().cpu().numpy()
    except Exception:
        pass
    return np.asarray(x)

def dict_to_matrix(d):
    if not isinstance(d, dict) or len(d) == 0:
        return np.empty((0,0), dtype=np.float32), []
    keys = list(d.keys())
    mats = []
    for k in keys:
        v = _to_numpy(d[k])
        if v.ndim == 1: v = v[None, :]
        mats.append(v.astype(np.float32, copy=False))
    return np.vstack(mats), keys

def stem(p):  # имя файла без расширения
    return Path(p).stem

# --- 0) шаблон сабмита ---
sub_df = pd.read_csv(SUBMISSION_IN)
N = len(sub_df)
print(f"[info] submission rows: {N}")

# --- 1) галерея: emb_stl_test / emb_img_test (или считаем на лету) ---
if "emb_stl_test" not in globals() or len(emb_stl_test) == 0:
    print("[info] emb_stl_test not found → compute from", GALLERY_STL_DIR)
    emb_stl_test = generate_stl_embeddings(stl_encoder, GALLERY_STL_DIR, NUM_POINTS, device)

if "emb_img_test" not in globals() or len(emb_img_test) == 0:
    print("[info] emb_img_test not found → compute from", GALLERY_IMG_DIR)
    emb_img_test = generate_image_embeddings(image_encoder, GALLERY_IMG_DIR, IMAGE_SIZE, device)

E_stl_gallery, stl_gallery_keys = dict_to_matrix(emb_stl_test)
E_img_gallery,  img_gallery_keys = dict_to_matrix(emb_img_test)
assert E_stl_gallery.size and E_img_gallery.size, "Empty gallery embeddings"
assert E_stl_gallery.shape[1] == E_img_gallery.shape[1], "Gallery dims mismatch"

# --- 2) эмбеддинги запросов ---
q_img_emb = generate_image_embeddings(image_encoder, QUERY_IMAGES_DIR, IMAGE_SIZE, device)
E_qimg, qimg_keys = dict_to_matrix(q_img_emb)
assert E_qimg.size, f"No query image embeddings from {QUERY_IMAGES_DIR}"

q_stl_emb = generate_stl_embeddings(stl_encoder, QUERY_MESHES_DIR, NUM_POINTS, device)
E_qstl, qstl_keys = dict_to_matrix(q_stl_emb)
assert E_qstl.size, f"No query mesh embeddings from {QUERY_MESHES_DIR}"

assert E_qimg.shape[1] == E_stl_gallery.shape[1], "D mismatch: query images vs mesh gallery"
assert E_qstl.shape[1] == E_img_gallery.shape[1], "D mismatch: query meshes vs image gallery"

# --- 3) image -> mesh: TOP-5 (без расширений, формат Python-списка через repr) ---
S_i2m = 1.0 - cdist(E_stl_gallery, E_qimg, metric="cosine")   # [M_gallery, N_qimg]
idx_i2m = np.argsort(-S_i2m, axis=0)[:5, :]

rows_i2m = []
for j in range(S_i2m.shape[1]):
    image_id = stem(qimg_keys[j])  # без расширения
    top_mesh_ids = [stem(stl_gallery_keys[i]) for i in idx_i2m[:, j]]
    rows_i2m.append({
        "image_id": image_id,
        "mesh_top5_str": repr(top_mesh_ids)  # выглядит как "['q0063','g0028',...]"
    })
df_i2m = pd.DataFrame(rows_i2m).sort_values("image_id").reset_index(drop=True)

# --- 4) mesh -> image: TOP-5 (без расширений, список через repr) ---
S_m2i = 1.0 - cdist(E_qstl, E_img_gallery, metric="cosine")   # [N_qstl, M_gallery_img]
idx_m2i = np.argsort(-S_m2i, axis=1)[:, :5]

rows_m2i = []
for i in range(S_m2i.shape[0]):
    mesh_id = stem(qstl_keys[i])  # без .stl
    top_image_ids = [stem(img_gallery_keys[j]) for j in idx_m2i[i]]
    rows_m2i.append({
        "mesh_id": mesh_id,
        "image_top5_str": repr(top_image_ids)
    })
df_m2i = pd.DataFrame(rows_m2i).sort_values("mesh_id").reset_index(drop=True)

# --- 5) заполняем сабмит: ВСЕ имена без расширений; массивы — строкой списка (repr) ---
filled = sub_df.copy()

# приводим к длине N
df_i2m = df_i2m.head(N)
df_m2i = df_m2i.head(N)

# image->mesh
filled.loc[:len(df_i2m)-1, "image_to_mesh_image"] = df_i2m["image_id"].values                     # без расширения
filled.loc[:len(df_i2m)-1, "image_to_mesh_mesh"]  = df_i2m["mesh_top5_str"].values               # "['..','..','..']"

# mesh->image
filled.loc[:len(df_m2i)-1, "mesh_to_image_mesh"]  = df_m2i["mesh_id"].values                     # без расширения
filled.loc[:len(df_m2i)-1, "mesh_to_image_image"] = df_m2i["image_top5_str"].values              # "['..','..','..']"

# --- 6) сохраняем ---
filled.to_csv(SUBMISSION_OUT, index=False)
print("Saved:", SUBMISSION_OUT)
display(filled.head(10))


[info] submission rows: 100


Генерация Image эмбеддингов:   0%|          | 0/100 [00:00<?, ?it/s]

Генерация STL эмбеддингов: 100%|██████████| 100/100 [00:15<00:00,  6.40it/s]


Saved: submission.csv


,id,image_to_mesh_image,image_to_mesh_mesh,mesh_to_image_mesh,mesh_to_image_image
0,0,q0000,"['g0031', 'g0072', 'g0033', 'g0038', 'g0099']",q0000,"['g0058_6', 'g0027_19', 'g0027_21', 'g0027_9',..."
1,1,q0001,"['g0039', 'g0047', 'g0014', 'g0007', 'g0085']",q0001,"['g0045_9', 'g0045_0', 'g0045_6', 'g0045_1', '..."
2,2,q0002,"['g0080', 'g0090', 'g0067', 'g0011', 'g0020']",q0002,"['g0079_20', 'g0079_12', 'g0079_0', 'g0079_3',..."
3,3,q0003,"['g0018', 'g0024', 'g0052', 'g0058', 'g0070']",q0003,"['g0050_18', 'g0029_18', 'g0050_15', 'g0012_9'..."
4,4,q0004,"['g0006', 'g0025', 'g0007', 'g0054', 'g0046']",q0004,"['g0076_7', 'g0030_6', 'g0030_0', 'g0030_9', '..."
5,5,q0005,"['g0057', 'g0023', 'g0074', 'g0071', 'g0002']",q0005,"['g0097_24', 'g0067_10', 'g0097_25', 'g0002_22..."
6,6,q0006,"['g0015', 'g0090', 'g0014', 'g0083', 'g0040']",q0006,"['g0010_3', 'g0010_19', 'g0010_9', 'g0010_23',..."
7,7,q0007,"['g0010', 'g0075', 'g0016', 'g0081', 'g0034']",q0007,"['g0093_6', 'g0093_11', 'g0046_17', 'g0093_8',..."
8,8,q0008,"['g0076', 'g0035', 'g0090', 'g0067', 'g0078']",q0008,"['g0003_18', 'g0056_2', 'g0003_2', 'g0003_19',..."
9,9,q0009,"['g0005', 'g0009', 'g0012', 'g0027', 'g0000']",q0009,"['g0095_4', 'g0015_20', 'g0060_10', 'g0015_16'..."
